In [1]:
import pandas as pd

df = pd.read_csv('../data/online_retail_II.csv')
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [2]:
print(df.shape)
df.info()

(1067371, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB


In [3]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [4]:
# Returns/cancellations show as negative quantities
print("Negative quantities:", (df['Quantity'] < 0).sum())

# Zero or negative prices
print("Zero/negative prices:", (df['Price'] <= 0).sum())

# Date range
print(df['InvoiceDate'].min(), "to", df['InvoiceDate'].max())

# How many unique customers do we actually have?
print("Unique customers:", df['Customer ID'].nunique())

Negative quantities: 22950
Zero/negative prices: 6207
2009-12-01 07:45:00 to 2011-12-09 12:50:00
Unique customers: 5942


In [5]:
# Do cancelled invoices have a marker?
print(df[df['Quantity'] < 0]['Invoice'].astype(str).str[0].value_counts())

Invoice
C    19493
5     2728
4      729
Name: count, dtype: int64


In [6]:
# Convert date column first
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Drop missing customer IDs
df = df.dropna(subset=['Customer ID'])

# Drop invalid prices
df = df[df['Price'] > 0]

# Identify cancellations
df['is_cancellation'] = df['Invoice'].astype(str).str.startswith('C')

# Drop unmarked negatives (inventory adjustments, not customer behaviour)
df = df[~((df['Quantity'] < 0) & (~df['is_cancellation']))]

# Create revenue column
df['Revenue'] = df['Quantity'] * df['Price']

print(df.shape)
print("Unique customers:", df['Customer ID'].nunique())

(824293, 10)
Unique customers: 5939


## Data Cleaning
Dropped 243,007 rows with missing Customer ID (RFM requires customer-level 
attribution) and 6,207 rows with invalid prices. Negative quantities were split: 
19,493 marked cancellations (invoice prefix 'C') were retained as genuine 
customer returns, while 3,457 unmarked negatives were dropped as inventory 
adjustments rather than customer behaviour. Final: 824,293 transactions across 
5,939 customers.

In [7]:
# Reload original to inspect what we dropped
raw = pd.read_csv('../data/online_retail_II.csv')

# Are missing Customer IDs concentrated anywhere?
missing = raw[raw['Customer ID'].isnull()]
print(missing['Country'].value_counts().head())
print("\nDate range of missing:", missing['InvoiceDate'].min(), "to", missing['InvoiceDate'].max())
print("\nAvg transaction value, missing vs present:")
print("Missing:", (missing['Quantity'] * missing['Price']).mean())
print("Present:", (raw[raw['Customer ID'].notna()]['Quantity'] * raw[raw['Customer ID'].notna()]['Price']).mean())

Country
United Kingdom    240029
EIRE                1671
Hong Kong            364
Unspecified          232
France               128
Name: count, dtype: int64

Date range of missing: 2009-12-01 10:52:00 to 2011-12-09 10:26:00

Avg transaction value, missing vs present:
Missing: 10.859597377853312
Present: 20.195317102639127


## Limitation: Non-Random Missingness
23% of transactions lacked a Customer ID and were dropped, since RFM requires 
customer-level attribution. This missingness is not random — dropped transactions 
averaged £10.86 versus £20.20 for identified customers, suggesting they represent 
guest checkouts or one-off purchases. The resulting segmentation therefore describes 
registered, identifiable customers rather than the full customer base. This is 
acceptable for the analysis goal (targeting known customers for retention), but 
segment sizes should not be read as representative of total purchasing activity.

In [8]:
df.to_csv('../data/retail_clean.csv', index=False)
print("Saved:", df.shape)

Saved: (824293, 10)


In [9]:
import os
print(os.listdir('../data'))

['online_retail_II.csv', 'retail_clean.csv', 'rfm.csv', 'rfm_segments.csv']
